"""
**PPMI Data Preprocessing - Stage 1: Raw Data Merger**

**Objective:**
Collect and merge key PPMI datasets into a unified tabular format for downstream analysis.

**Input Data:**
- MRI (FreeSurfer: cortical thickness, subcortical volumes)
- DAT-SPECT (striatal binding ratios)
- UPDRS (Part III motor scores)
- Demographics (age, sex, education)
- Genotype (APOE, SNCA, etc.)

**Processing Steps:**
1. Load raw CSV files from PPMI
2. Select clinically relevant features
3. Standardize patient IDs and visit labels (BL/SC)
4. Left-merge datasets on PATNO to retain all available data

**Output:**
- Raw merged dataset (PPMI_Merged_Raw.csv) with missing values preserved

**Next Steps (Stage 2):**
- Advanced imputation (MICE)
- Feature engineering
- Outlier handling
"""

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load each dataset
demographics = pd.read_csv('Demographics_10Jun2025.csv')          # Age, sex, education
datscan = pd.read_csv('DaTScan_SBR_Analysis_07Jun2025.csv')                # Striatal binding ratios
updrs = pd.read_csv('MDS-UPDRS_Part_III_16May2025.csv')          # Motor scores
mri_ct = pd.read_csv('FS7_APARC_CTH_07Jun2025.csv')               # Cortical thickness
mri_vol = pd.read_csv('FS7_ASEG_VOL_07Jun2025.csv')               # Subcortical volumes
age_visit = pd.read_csv('Age_at_visit_10Jun2025 (1).csv')


<ipython-input-4-3882453910>:4: DtypeWarning: Columns (15,19) have mixed types. Specify dtype option on import or set low_memory=False.
  updrs = pd.read_csv('MDS-UPDRS_Part_III_16May2025.csv')          # Motor scores


In [ ]:
imputed_data1 = pd.read_csv("ppmi_clean_imputed (1).csv")             # Age per visit

Cleaning each data.


In [ ]:
demographics.head()

,REC_ID,PATNO,EVENT_ID,PAG_NAME,INFODT,AFICBERB,ASHKJEW,BASQUE,BIRTHDT,SEX,...,HISPLAT,RAASIAN,RABLACK,RAHAWOPI,RAINDALS,RANOS,RAWHITE,RAUNKNOWN,ORIG_ENTRY,LAST_UPDATE
0,IA86904,3000,TRANS,SCREEN,01/2011,0.0,0.0,0.0,12/1941,0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,01/2011,2022-11-07 00:00:00.0
1,IA86905,3001,TRANS,SCREEN,02/2011,0.0,0.0,0.0,01/1946,1,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,02/2011,2022-11-07 00:00:00.0
2,IA86906,3002,TRANS,SCREEN,03/2011,0.0,0.0,0.0,08/1943,0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,03/2011,2022-11-07 00:00:00.0
3,IA86907,3003,TRANS,SCREEN,03/2011,0.0,0.0,0.0,07/1954,0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,03/2011,2022-11-07 00:00:00.0
4,IA86908,3004,TRANS,SCREEN,03/2011,0.0,0.0,0.0,11/1951,1,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,03/2011,2022-11-07 00:00:00.0


In [ ]:
demo_clean = demographics[['PATNO', 'SEX']].drop_duplicates()

In [ ]:
demo_clean.head()

,PATNO,SEX
0,3000,0
1,3001,1
2,3002,0
3,3003,0
4,3004,1


In [ ]:
age_visit.head(60)

,PATNO,EVENT_ID,AGE_AT_VISIT
0,3000,BL,69.1
1,3000,R17,80.5
2,3000,R18,81.4
3,3000,BL,69.1
4,3000,V01,69.4
5,3000,V02,69.6
6,3000,V03,69.9
7,3000,V04,70.2
8,3000,V05,70.6
9,3000,V06,71.2


In [ ]:
def get_baseline_age(age_df):
    """
    Returns the most appropriate baseline age per patient:
    1. Prefer official BL age when available
    2. Fall back to SC age when BL is missing
    3. Flag cases where SC was used
    """
    # Standardize visit labels
    age_df['EVENT_ID'] = age_df['EVENT_ID'].replace({'SC': 'BL'})  # Treat SC as BL

    # Get best available age per patient
    baseline_ages = (
        age_df[age_df['EVENT_ID'] == 'BL']
        .sort_values(['PATNO', 'AGE_AT_VISIT'])  # Break ties consistently
        .drop_duplicates('PATNO', keep='first')
        [['PATNO', 'AGE_AT_VISIT']]
        .rename(columns={'AGE_AT_VISIT': 'AGE'})
    )

    return baseline_ages

# Usage
baseline_ages = get_baseline_age(age_visit)
print(f"Baseline ages extracted for {len(baseline_ages)} patients")

Baseline ages extracted for 5087 patients


In [ ]:
baseline_ages.head(50)

,PATNO,AGE
0,3000,69.1
20,3001,65.1
44,3002,67.6
68,3003,56.6
93,3004,59.3
112,3006,57.4
117,3007,64.5
122,3008,81.8
145,3009,83.6
169,3010,46.9


In [ ]:
baseline_ages.tail(50)

,PATNO,AGE
36364,408874,48.9
36365,408912,64.1
36366,409073,65.0
36368,411324,64.4
36369,411347,61.6
36370,411443,67.5
36372,411621,57.2
36373,411631,69.8
36375,411854,70.4
36377,412845,67.4


In [ ]:
# 1. Convert scan date to datetime for proper sorting
datscan['DATSCAN_DATE'] = pd.to_datetime(datscan['DATSCAN_DATE'], format='%m/%Y')

# 2. Sort by patient and scan date (earliest first)
datscan = datscan.sort_values(['PATNO', 'DATSCAN_DATE'])

# 4. Keep only the first scan per patient (baseline)
baseline_dat = (
    datscan.groupby('PATNO', as_index=False)
    .first()  # Takes first row per PATNO (already sorted by date)
)

# 5. Select and rename key columns
cols_to_keep = {
    'PATNO': 'PATNO',
    'EVENT_ID': 'EVENT_ID',  # Preserve original visit label
    'DATSCAN_CAUDATE_L': 'CAUDATE_L',
    'DATSCAN_CAUDATE_R': 'CAUDATE_R',
    'DATSCAN_PUTAMEN_L': 'PUTAMEN_L',
    'DATSCAN_PUTAMEN_R': 'PUTAMEN_R',
    'DATSCAN_DATE': 'SCAN_DATE'
}

baseline_dat = baseline_dat[list(cols_to_keep.keys())].rename(columns=cols_to_keep)

# 6. Verify
print(f"Final baseline scans by visit type:\n{baseline_dat['EVENT_ID'].value_counts()}")
print(f"\nSample data:\n{baseline_dat.head()}")

Final baseline scans by visit type:
EVENT_ID
SC      2066
U01       34
V04       17
SC99      16
V06        5
V10        2
Name: count, dtype: int64

Sample data:
   PATNO EVENT_ID  CAUDATE_L  CAUDATE_R  PUTAMEN_L  PUTAMEN_R  SCAN_DATE
0   3000       SC       3.43       2.99       2.63       2.94 2011-01-01
1   3001      U01       1.92       2.00       0.65       0.80 2011-06-01
2   3002      U01       3.72       2.92       1.78       1.01 2011-06-01
3   3003      U01       2.54       3.63       0.68       1.11 2011-08-01
4   3004      U01       5.30       5.09       2.97       3.54 2011-08-01


In [ ]:
baseline_dat['EVENT_ID'] = 'BL'


In [ ]:
baseline_dat.shape

(2140, 7)

In [ ]:
baseline_dat.head()

,PATNO,EVENT_ID,CAUDATE_L,CAUDATE_R,PUTAMEN_L,PUTAMEN_R,SCAN_DATE
0,3000,BL,3.43,2.99,2.63,2.94,2011-01-01
1,3001,BL,1.92,2.00,0.65,0.80,2011-06-01
2,3002,BL,3.72,2.92,1.78,1.01,2011-06-01
3,3003,BL,2.54,3.63,0.68,1.11,2011-08-01
4,3004,BL,5.30,5.09,2.97,3.54,2011-08-01


**There is no negligible difference between "SC and "BC" so going forward we can keet SC for the DAT-SPECT data and later merge with BL on MRI of genotype**

In [ ]:
mri_ct.head()

,PATNO,EVENT_ID,lh_bankssts,lh_caudalanteriorcingulate,lh_caudalmiddlefrontal,lh_cuneus,lh_entorhinal,lh_fusiform,lh_inferiorparietal,lh_inferiortemporal,...,rh_rostralmiddlefrontal,rh_superiorfrontal,rh_superiorparietal,rh_superiortemporal,rh_supramarginal,rh_frontalpole,rh_temporalpole,rh_transversetemporal,rh_insula,rh_MeanThickness
0,3000,BL,2.487,2.778,2.407,1.714,2.805,2.693,2.183,2.428,...,2.234,2.503,2.024,2.542,2.216,2.402,4.123,2.293,2.848,2.32243
1,3001,BL,2.241,2.169,2.253,1.642,3.539,2.607,2.283,2.626,...,2.064,2.390,2.062,2.556,2.231,2.486,3.840,2.271,2.756,2.28570
2,3002,BL,2.475,2.265,2.606,1.832,3.771,2.956,2.542,2.747,...,2.359,2.670,2.109,2.534,2.561,2.530,3.501,2.545,3.040,2.45438
3,3003,BL,2.547,2.551,2.519,1.874,3.922,2.558,2.432,2.993,...,2.291,2.701,2.166,2.647,2.399,2.496,3.946,2.756,2.909,2.41209
4,3004,BL,2.717,2.625,2.513,1.741,3.494,2.699,2.421,2.613,...,2.254,2.631,2.242,2.670,2.250,2.511,3.568,2.356,2.929,2.39909


In [ ]:
mri_vol.head()

,PATNO,EVENT_ID,Left_WM_hypointensities,Brain_Stem,Left_non_WM_hypointensities,Optic_Chiasm,Right_WM_hypointensities,BrainSegVol,Right_Lateral_Ventricle,CC_Central,...,Left_VentralDC,SupraTentorialVolNotVent,CC_Mid_Anterior,SupraTentorialVol,SubCortGrayVol,Right_Thalamus,Left_Lateral_Ventricle,CSF,SurfaceHoles,CerebralWhiteMatterVol
0,3000,BL,0,20725.6,0,96.7,0,1144050,9579.3,936.4,...,4188.8,984766,599.2,1010680,52729.0,7122.1,9970.7,1039.0,23,497730
1,3001,BL,0,23802.3,0,152.0,0,1297010,17897.3,887.0,...,4407.5,1094140,542.6,1142600,67357.0,7495.3,22811.9,1481.8,26,565148
2,3002,BL,0,19271.1,0,102.1,0,1040700,8822.4,568.9,...,3191.4,888974,725.7,910643,54813.0,6470.1,8445.8,835.3,23,425313
3,3003,BL,0,22191.0,0,107.3,0,1177320,8620.4,1020.8,...,4427.0,1010570,1279.4,1033260,60577.0,7458.9,9266.9,1235.7,16,510845
4,3004,BL,0,24473.0,0,135.3,0,1211960,9447.4,857.1,...,3837.1,1058590,831.4,1080080,56642.0,6451.1,6889.7,851.9,21,540211


In [ ]:
mri_vol.columns

Index(['PATNO', 'EVENT_ID', 'Left_WM_hypointensities', 'Brain_Stem',
       'Left_non_WM_hypointensities', 'Optic_Chiasm',
       'Right_WM_hypointensities', 'BrainSegVol', 'Right_Lateral_Ventricle',
       'CC_Central', '5th_Ventricle', 'Right_choroid_plexus',
       'Right_Cerebellum_White_Matter', 'Left_vessel',
       'Left_Cerebellum_Cortex', 'MaskVol_to_eTIV', 'MaskVol', 'TotalGrayVol',
       'Left_choroid_plexus', 'Right_Inf_Lat_Vent', 'Left_Pallidum',
       'Left_Thalamus', 'Right_VentralDC', 'rhCortexVol',
       'Right_non_WM_hypointensities', 'BrainSegVol_to_eTIV', 'Right_Amygdala',
       'Left_Amygdala', 'EstimatedTotalIntraCranialVol', '4th_Ventricle',
       'Left_Inf_Lat_Vent', 'CortexVol', 'Right_Pallidum', 'lhCortexVol',
       'CC_Anterior', 'CC_Posterior', 'Left_Accumbens_area', 'Right_vessel',
       'Right_Cerebellum_Cortex', 'Left_Putamen', '3rd_Ventricle',
       'non_WM_hypointensities', 'Right_Caudate', 'CC_Mid_Posterior',
       'lhSurfaceHoles', 'Left_Hipp

In [ ]:
vol_selected = [
    'PATNO', 'EVENT_ID',

    # Subcortical motor/cognitive structures
    'Left_Putamen', 'Right_Putamen',
    'Left_Caudate', 'Right_Caudate',
    'Left_Pallidum', 'Right_Pallidum',
    'Left_Thalamus', 'Right_Thalamus',
    'Left_Accumbens_area', 'Right_Accumbens_area',
    'Left_Hippocampus', 'Right_Hippocampus',
    'Left_Amygdala', 'Right_Amygdala',

    # Global brain volumes
    'BrainSegVol', 'BrainSegVol_to_eTIV',
    'EstimatedTotalIntraCranialVol', 'CortexVol',
    'TotalGrayVol', 'SubCortGrayVol', 'CerebralWhiteMatterVol'
]
mri_vol_clean = mri_vol[vol_selected]

In [ ]:
mri_vol_clean.head()

,PATNO,EVENT_ID,Left_Putamen,Right_Putamen,Left_Caudate,Right_Caudate,Left_Pallidum,Right_Pallidum,Left_Thalamus,Right_Thalamus,...,Right_Hippocampus,Left_Amygdala,Right_Amygdala,BrainSegVol,BrainSegVol_to_eTIV,EstimatedTotalIntraCranialVol,CortexVol,TotalGrayVol,SubCortGrayVol,CerebralWhiteMatterVol
0,3000,BL,4380.2,3956.8,2838.1,3154.0,1765.6,1728.2,6955.3,7122.1,...,4475.3,960.7,1687.3,1144050,0.725653,1576580,436006,600024,52729.0,497730
1,3001,BL,7369.2,6070.9,5282.2,5118.5,2268.1,2228.1,7746.8,7495.3,...,4549.5,1619.5,2077.7,1297010,0.752404,1723820,463326,651817,67357.0,565148
2,3002,BL,5060.8,5537.5,3740.9,3617.8,1913.5,2258.0,6805.1,6470.1,...,3655.6,1128.0,1666.0,1040700,0.761920,1365890,409747,572593,54813.0,425313
3,3003,BL,4832.5,6072.6,3128.6,3600.1,2092.5,2138.3,7451.9,7458.9,...,4837.2,1407.1,1864.0,1177320,0.742722,1585140,440234,616339,60577.0,510845
4,3004,BL,5383.9,5349.4,3074.5,3318.6,1393.8,1867.2,8138.2,6451.1,...,4183.5,1562.8,1920.1,1211960,0.743601,1629860,463672,623995,56642.0,540211


In [ ]:
mri_ct.columns

Index(['PATNO', 'EVENT_ID', 'lh_bankssts', 'lh_caudalanteriorcingulate',
       'lh_caudalmiddlefrontal', 'lh_cuneus', 'lh_entorhinal', 'lh_fusiform',
       'lh_inferiorparietal', 'lh_inferiortemporal', 'lh_isthmuscingulate',
       'lh_lateraloccipital', 'lh_lateralorbitofrontal', 'lh_lingual',
       'lh_medialorbitofrontal', 'lh_middletemporal', 'lh_parahippocampal',
       'lh_paracentral', 'lh_parsopercularis', 'lh_parsorbitalis',
       'lh_parstriangularis', 'lh_pericalcarine', 'lh_postcentral',
       'lh_posteriorcingulate', 'lh_precentral', 'lh_precuneus',
       'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal',
       'lh_superiorfrontal', 'lh_superiorparietal', 'lh_superiortemporal',
       'lh_supramarginal', 'lh_frontalpole', 'lh_temporalpole',
       'lh_transversetemporal', 'lh_insula', 'lh_MeanThickness', 'rh_bankssts',
       'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
       'rh_entorhinal', 'rh_fusiform', 'rh_inferiorparietal',
    

In [ ]:
ct_selected = [
    'PATNO', 'EVENT_ID',

    # Early Braak regions
    'lh_entorhinal', 'rh_entorhinal',
    'lh_parahippocampal', 'rh_parahippocampal',

    # Motor cortex
    'lh_precentral', 'rh_precentral',
    'lh_paracentral', 'rh_paracentral',

    # Temporal & fusiform (cognitive)
    'lh_middletemporal', 'rh_middletemporal',
    'lh_inferiortemporal', 'rh_inferiortemporal',
    'lh_fusiform', 'rh_fusiform',

    # DMN-related
    'lh_posteriorcingulate', 'rh_posteriorcingulate',
    'lh_precuneus', 'rh_precuneus',

    # Frontal/limbic
    'lh_medialorbitofrontal', 'rh_medialorbitofrontal',
    'lh_rostralanteriorcingulate', 'rh_rostralanteriorcingulate',

    # Global cortical thickness
    'lh_MeanThickness', 'rh_MeanThickness'
]
mri_ct_clean = mri_ct[ct_selected]

In [ ]:
mri_ct_clean.head()

,PATNO,EVENT_ID,lh_entorhinal,rh_entorhinal,lh_parahippocampal,rh_parahippocampal,lh_precentral,rh_precentral,lh_paracentral,rh_paracentral,...,lh_posteriorcingulate,rh_posteriorcingulate,lh_precuneus,rh_precuneus,lh_medialorbitofrontal,rh_medialorbitofrontal,lh_rostralanteriorcingulate,rh_rostralanteriorcingulate,lh_MeanThickness,rh_MeanThickness
0,3000,BL,2.805,3.086,2.538,2.924,2.411,2.327,2.500,2.407,...,2.492,2.522,2.306,2.223,2.405,2.292,2.975,2.848,2.30500,2.32243
1,3001,BL,3.539,3.611,2.533,2.449,2.279,2.316,2.160,2.033,...,2.474,2.482,2.341,2.335,2.458,2.298,2.645,2.520,2.31882,2.28570
2,3002,BL,3.771,3.822,2.684,3.019,2.422,2.514,2.342,2.389,...,2.366,2.362,2.311,2.193,2.270,2.429,2.860,2.515,2.48731,2.45438
3,3003,BL,3.922,3.583,2.894,2.792,2.427,2.345,2.315,2.515,...,2.691,2.782,2.415,2.656,2.439,2.477,2.639,2.544,2.47113,2.41209
4,3004,BL,3.494,3.380,2.800,2.738,2.619,2.509,2.281,2.281,...,2.567,2.225,2.286,2.454,2.409,2.268,2.881,2.476,2.46058,2.39909


In [ ]:
print("mri_vol shape", mri_vol_clean.shape)

mri_vol shape (1713, 23)


In [ ]:
print("mri_ct shape", mri_ct_clean.shape)

mri_ct shape (1716, 26)


In [ ]:
updrs.head(50)

,REC_ID,PATNO,EVENT_ID,PAG_NAME,INFODT,PDTRTMNT,PDSTATE,HRPOSTMED,HRDBSON,HRDBSOFF,...,NP3RTARL,NP3RTALL,NP3RTALJ,NP3RTCON,NP3TOT,DYSKPRES,DYSKIRAT,NHY,ORIG_ENTRY,LAST_UPDATE
0,272451901,3000,BL,NUPDRS3,02/2011,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,4.0,0.0,NaN,0.0,02/2011,2020-06-25 16:02:19.0
1,338703101,3000,V04,NUPDRS3,03/2012,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,0.0,NaN,0.0,03/2012,2020-06-25 16:02:22.0
2,385009801,3000,V06,NUPDRS3,02/2013,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,4.0,0.0,NaN,0.0,02/2013,2020-06-25 16:02:22.0
3,437131401,3000,V08,NUPDRS3,03/2014,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,2.0,0.0,NaN,0.0,05/2014,2020-06-25 16:02:22.0
4,512469901,3000,V10,NUPDRS3,03/2015,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,19.0,0.0,NaN,0.0,03/2015,2020-06-25 16:02:23.0
5,563731101,3000,V12,NUPDRS3,04/2016,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,3.0,0.0,NaN,0.0,04/2016,2020-06-25 16:02:23.0
6,675890201,3000,V14,NUPDRS3,02/2018,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,10.0,0.0,NaN,0.0,02/2018,2020-06-25 16:02:24.0
7,736290801,3000,V15,NUPDRS3,03/2019,0.0,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,5.0,0.0,NaN,0.0,04/2019,2020-06-25 16:02:24.0
8,IANT161818,3000,V17,NUPDRDOSE3,05/2021,0.0,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,06/2021,2021-06-01 00:00:00.0
9,278743601,3001,BL,NUPDRS3,03/2011,0.0,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,12.0,0.0,NaN,1.0,03/2011,2021-07-31 00:00:00.0


In [ ]:
updrs.shape

(33067, 63)

In [ ]:
# Keep only BL records (ignore SC/Vxx visits)
updrs_bl = updrs[updrs['EVENT_ID'] == 'BL'].copy()

In [ ]:
updrs_bl.shape

(4332, 63)

In [ ]:
cols_to_keep = ['PATNO', 'EVENT_ID', 'NP3TOT', 'NHY']  # Motor score + Hoehn & Yahr stage
updrs_bl = updrs_bl[cols_to_keep]

In [ ]:
print(f"Unique patients: {updrs_bl['PATNO'].nunique()}")
print(f"UPDRS-III score range: {updrs_bl['NP3TOT'].min()} to {updrs_bl['NP3TOT'].max()}")

Unique patients: 4141
UPDRS-III score range: 0.0 to 71.0


In [ ]:
updrs_bl.head(50)

,PATNO,EVENT_ID,NP3TOT,NHY
0,3000,BL,4.0,0.0
9,3001,BL,12.0,1.0
37,3002,BL,17.0,2.0
63,3003,BL,29.0,2.0
88,3004,BL,2.0,0.0
100,3006,BL,22.0,2.0
107,3007,BL,21.0,2.0
109,3008,BL,10.0,0.0
118,3009,BL,6.0,0.0
128,3010,BL,19.0,2.0


In [ ]:
imputed_data1.head()

,PATNO,EVENT_ID_x,NP1RTOT,NP2PTOT,NP3TOT,NP1DDS,NP1APAT,NP2WALK,NP2FREZ,NP3BRADY,...,caudalanteriorcingulate_avg,posteriorcingulate_avg,entorhinal_avg,parahippocampal_avg,insula_avg,supramarginal_avg,COHORT_DEFINITION,EVENT_ID_y,AGE_AT_VISIT,APOE4
0,3000,BL,3.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,656.5,902.0,513.0,713.0,2522.0,3805.5,Healthy Control,BL,69.1,0
1,3001,BL,0.0,2.0,12.0,0.0,0.0,0.0,0.0,1.0,...,756.5,857.5,510.0,767.5,2550.5,4355.0,Parkinson's Disease,BL,65.1,0
2,3002,BL,3.0,15.0,17.0,0.0,1.0,1.0,1.0,2.0,...,503.0,948.5,417.0,581.0,2083.5,3015.0,Parkinson's Disease,BL,67.6,0
3,3003,BL,1.0,6.0,29.0,0.0,0.0,1.0,0.0,2.0,...,394.0,804.0,416.0,659.5,2352.5,3477.5,Parkinson's Disease,BL,56.7,1
4,3004,BL,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,576.5,1085.5,509.0,697.0,2404.5,4814.0,Healthy Control,BL,59.4,0


In [ ]:
imputed_data_clean = imputed_data1[["PATNO","APOE4","COHORT_DEFINITION","EVENT_ID_x"]]

In [ ]:
imputed_data_clean.head()

,PATNO,APOE4,COHORT_DEFINITION,EVENT_ID_x
0,3000,0,Healthy Control,BL
1,3001,0,Parkinson's Disease,BL
2,3002,0,Parkinson's Disease,BL
3,3003,1,Parkinson's Disease,BL
4,3004,0,Healthy Control,BL


In [ ]:
from functools import reduce

def safe_merge(left, right):
    return pd.merge(left, right, on='PATNO', how='left')

# Drop duplicates before merging and ensure PATNO is string type
datasets = [
    demo_clean.drop_duplicates('PATNO').astype({'PATNO': str}),
    baseline_ages.drop_duplicates('PATNO').astype({'PATNO': str}),
    baseline_dat.drop_duplicates('PATNO').astype({'PATNO': str}),
    mri_vol.drop_duplicates('PATNO').astype({'PATNO': str}),
    mri_ct.drop_duplicates('PATNO').astype({'PATNO': str}),
    updrs_bl.drop_duplicates('PATNO').astype({'PATNO': str}),
    imputed_data_clean.drop_duplicates('PATNO')[['PATNO', 'APOE4', 'COHORT_DEFINITION']].astype({'PATNO': str})
]

final_df = reduce(safe_merge, datasets)

In [ ]:
final_df.shape


(5281, 147)

In [ ]:
final_df.head(50)

,PATNO,SEX,AGE,CAUDATE_L,CAUDATE_R,PUTAMEN_L,PUTAMEN_R,EVENT_ID_x,Left_WM_hypointensities,Brain_Stem,...,rh_frontalpole,rh_temporalpole,rh_transversetemporal,rh_insula,rh_MeanThickness,NP3TOT,NHY,APOE4,COHORT_DEFINITION,missing_features
0,3000,0,69.1,3.43,2.99,2.63,2.94,BL,0.0,20725.6,...,2.402,4.123,2.293,2.848,2.32243,4.0,0.0,0.0,Healthy Control,0
1,3001,1,65.1,1.92,2.00,0.65,0.80,BL,0.0,23802.3,...,2.486,3.840,2.271,2.756,2.28570,12.0,1.0,0.0,Parkinson's Disease,0
2,3002,0,67.6,3.72,2.92,1.78,1.01,BL,0.0,19271.1,...,2.530,3.501,2.545,3.040,2.45438,17.0,2.0,0.0,Parkinson's Disease,0
3,3003,0,56.6,2.54,3.63,0.68,1.11,BL,0.0,22191.0,...,2.496,3.946,2.756,2.909,2.41209,29.0,2.0,1.0,Parkinson's Disease,0
4,3004,1,59.3,5.30,5.09,2.97,3.54,BL,0.0,24473.0,...,2.511,3.568,2.356,2.929,2.39909,2.0,0.0,0.0,Healthy Control,0
6,3006,0,57.4,2.12,2.28,0.15,0.76,BL,0.0,22821.2,...,2.670,3.646,2.467,3.016,2.47456,22.0,2.0,1.0,Parkinson's Disease,0
7,3007,1,64.5,NaN,NaN,NaN,NaN,BL,0.0,24603.1,...,2.468,3.686,2.302,2.794,2.28610,21.0,2.0,0.0,Parkinson's Disease,4
8,3008,0,81.8,3.46,3.99,2.12,2.07,BL,0.0,17679.7,...,2.335,2.932,1.865,2.622,2.10820,10.0,0.0,0.0,Healthy Control,0
10,3010,1,46.9,3.65,2.86,1.24,0.45,BL,0.0,23738.9,...,2.155,4.137,2.041,3.031,2.32332,19.0,2.0,0.0,Parkinson's Disease,0
11,3011,1,31.8,3.89,3.69,2.74,3.27,BL,0.0,19136.4,...,2.514,3.800,2.440,3.091,2.51135,0.0,0.0,0.0,Healthy Control,0


In [ ]:
from google.colab import files

# Save to CSV
final_df.to_csv('merged_ppmi_dataset.csv', index=False)

# Download
files.download('merged_ppmi_dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>